# Phase 2 — Acoustic Feature Extraction

**Input:** `outputs/merged_dataset.parquet` → `audio_path` column  
**Output:** `outputs/audio_features.parquet`

## Two representations per track

| Representation | Method | Dims | Purpose |
|---|---|---|---|
| `audio_handcrafted` | librosa spectral/timbral/rhythmic features | 141 | Interpretable ablation |
| `audio_clap` | CLAP `laion/larger_clap_music` embedding | 512 | Semantic deep embedding |

## Performance notes
- Audio is loaded **once** at 48 kHz (CLAP native rate), then resampled to 22 050 Hz for librosa
- The `tonnetz` bottleneck is removed: computed from `chroma_stft` not harmonic separation
- `beat_track` replaced by `tempo` estimation (avoids the expensive O(T²) tracking step)
- After numba JIT warmup (first track): ~0.65 s/track hand-crafted + ~0.94 s/track CLAP
- Checkpoints every 100 tracks → safe to interrupt and resume

**Estimated wall time: ~90 min sequential on CPU**

In [ ]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import librosa
import librosa.feature.rhythm as lfr
import torch
from transformers import ClapModel, ClapProcessor
from tqdm import tqdm          # plain text bar — no ipywidgets required

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR   = os.path.join(ROOT, 'outputs')
CKPT_PATH = os.path.join(OUT_DIR, 'audio_features_checkpoint.pkl')
OUT_PATH  = os.path.join(OUT_DIR, 'audio_features.parquet')

SR_CLAP = 48_000   # CLAP native sample rate
SR_LIB  = 22_050   # librosa conventional sample rate
DURATION = 30      # seconds of audio to use per track
CKPT_EVERY = 100   # save checkpoint every N tracks

print('PyTorch:', torch.__version__)
print('Device: CPU (CLAP runs in fp32 inference mode)')
print('Output:', OUT_PATH)

## 1. Load Dataset

In [ ]:
df = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))

# Only process tracks that have audio
audio_df = df[df['has_audio']].reset_index(drop=True)
print(f'Tracks with audio: {len(audio_df)}')
print(f'Quadrant breakdown:')
print(audio_df['quadrant'].value_counts().sort_index().to_string())

## 2. Load CLAP Model (one-time, ~30 s)

In [ ]:
import time
print('Loading laion/larger_clap_music ...', end=' ', flush=True)
t0 = time.time()
clap_model     = ClapModel.from_pretrained('laion/larger_clap_music')
clap_processor = ClapProcessor.from_pretrained('laion/larger_clap_music')
clap_model.eval()
print(f'done in {time.time()-t0:.1f}s')
print(f'CLAP embedding dim: {clap_model.config.projection_dim}')

## 3. Feature Extraction Functions

In [ ]:
def agg(arr):
    """Collapse a (n_features, n_frames) matrix to mean+std → 2*n_features dims."""
    return np.concatenate([arr.mean(axis=1), arr.std(axis=1)])


def extract_handcrafted(y_lib, sr=SR_LIB):
    """
    Extract 141-dim hand-crafted acoustic feature vector.

    Group             | Features           | Agg       | Dims
    ------------------|--------------------|-----------|-----
    Timbral           | MFCCs (40)         | mean+std  |  80
    Harmonic          | Chroma (12)        | mean+std  |  24
    Harmonic (tonal)  | Tonnetz (6)        | mean+std  |  12
    Spectral          | Centroid           | mean+std  |   2
    Spectral          | Bandwidth          | mean+std  |   2
    Spectral          | Rolloff            | mean+std  |   2
    Spectral          | Contrast (7 bands) | mean+std  |  14
    Temporal          | ZCR                | mean+std  |   2
    Energy            | RMS                | mean+std  |   2
    Rhythmic          | Tempo (BPM)        | scalar    |   1
    Total             |                    |           | 141
    """
    # Pre-compute shared representations once to avoid redundant FFTs
    D   = np.abs(librosa.stft(y_lib))
    mel = librosa.feature.melspectrogram(y=y_lib, sr=sr, n_mels=128)

    mfcc     = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=40)
    chroma   = librosa.feature.chroma_stft(S=D, sr=sr)
    tonnetz  = librosa.feature.tonnetz(chroma=chroma)          # no harmonic step
    centroid = librosa.feature.spectral_centroid(S=D, sr=sr)
    bw       = librosa.feature.spectral_bandwidth(S=D, sr=sr)
    rolloff  = librosa.feature.spectral_rolloff(S=D, sr=sr)
    contrast = librosa.feature.spectral_contrast(S=D, sr=sr)
    zcr      = librosa.feature.zero_crossing_rate(y_lib)
    rms      = librosa.feature.rms(y=y_lib)
    onset    = librosa.onset.onset_strength(y=y_lib, sr=sr)
    tempo    = lfr.tempo(onset_envelope=onset, sr=sr)          # scalar BPM

    vec = np.concatenate([
        agg(mfcc),        # 80
        agg(chroma),      # 24
        agg(tonnetz),     # 12
        agg(centroid),    #  2
        agg(bw),          #  2
        agg(rolloff),     #  2
        agg(contrast),    # 14
        agg(zcr),         #  2
        agg(rms),         #  2
        np.atleast_1d(tempo).ravel()[:1],  #  1
    ])
    assert vec.shape == (141,), f'Expected 141 dims, got {vec.shape}'
    return vec.astype(np.float32)


def extract_clap(y_clap, sr=SR_CLAP):
    """
    Extract CLAP embeddings using laion/larger_clap_music.
    y_clap must be at 48 kHz.

    Returns
    -------
    clap_normalized : np.ndarray, shape (512,)
        L2-normalized projection — used for the audio-only FAISS index and late fusion.
    clap_raw : np.ndarray, shape (512,)
        Raw (un-normalized) projection — used for early fusion so that the concatenated
        [clap_raw; sbert_raw] has non-trivial combined scale and does not degenerate
        into equal-weight late fusion (which happens when both streams are unit-normalized
        before concatenation).
    """
    inputs = clap_processor(audio=y_clap, sampling_rate=sr, return_tensors='pt')
    with torch.no_grad():
        audio_outputs = clap_model.audio_model(
            input_features=inputs['input_features'],
            is_longer=inputs.get('is_longer'),
        )
        pooled     = audio_outputs[1]                                   # pooler_output
        raw        = clap_model.audio_projection(pooled)                # (1, 512), un-normalized
        normalized = raw / raw.norm(p=2, dim=-1, keepdim=True)         # (1, 512), L2-normalized
    return (
        normalized.squeeze(0).numpy().astype(np.float32),
        raw.squeeze(0).numpy().astype(np.float32),
    )


print('Feature extraction functions defined.')
print('  extract_handcrafted → 141-dim float32 vector')
print('  extract_clap        → (clap_normalized 512-dim, clap_raw 512-dim)')

In [ ]:
# Smoke-test both functions on one track
test_path = audio_df.loc[0, 'audio_path']
print(f'Smoke-test on: {os.path.basename(test_path)}')

y_48, _ = librosa.load(test_path, sr=SR_CLAP, mono=True, duration=DURATION)
y_22    = librosa.resample(y_48, orig_sr=SR_CLAP, target_sr=SR_LIB)

hc            = extract_handcrafted(y_22)
clp, clp_raw  = extract_clap(y_48)

print(f'  handcrafted:    shape={hc.shape}, any NaN={np.isnan(hc).any()}, range=[{hc.min():.3f}, {hc.max():.3f}]')
print(f'  clap_norm:      shape={clp.shape}, any NaN={np.isnan(clp).any()}, norm={np.linalg.norm(clp):.6f}')
print(f'  clap_raw:       shape={clp_raw.shape}, any NaN={np.isnan(clp_raw).any()}, norm={np.linalg.norm(clp_raw):.4f}')
print(f'  clp_raw is NOT unit-normalized: {abs(np.linalg.norm(clp_raw) - 1.0) > 0.01}')

## 4. Main Extraction Loop

Saves a checkpoint to `outputs/audio_features_checkpoint.pkl` every 100 tracks.  
**If interrupted, re-run this cell — it will skip already-processed tracks.**

In [ ]:
# Load checkpoint if it exists — but reject it if it predates the audio_clap_raw field.
# The old checkpoint stored only audio_clap (normalized); early fusion requires audio_clap_raw.
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH, 'rb') as fh:
        results = pickle.load(fh)
    if results and 'audio_clap_raw' not in results[0]:
        print('Checkpoint is outdated (missing audio_clap_raw). Restarting fresh.')
        results = []
    else:
        print(f'Resuming from checkpoint: {len(results)} tracks already processed')
else:
    results = []
    print('Starting fresh extraction')

done_ids = {r['song_id'] for r in results}

# Filter to unprocessed tracks
todo = audio_df[~audio_df['song_id'].isin(done_ids)].reset_index(drop=True)
print(f'Tracks remaining: {len(todo)}')

errors = []

for idx, row in tqdm(todo.iterrows(), total=len(todo), desc='Extracting audio features'):
    path = row['audio_path']
    try:
        # Load audio once at 48 kHz, resample for librosa
        y_48 = librosa.load(path, sr=SR_CLAP, mono=True, duration=DURATION)[0]
        y_22 = librosa.resample(y_48, orig_sr=SR_CLAP, target_sr=SR_LIB)

        hc, (clp, clp_raw) = extract_handcrafted(y_22), extract_clap(y_48)

        results.append({
            'song_id':           row['song_id'],
            'audio_handcrafted': hc.tolist(),
            'audio_clap':        clp.tolist(),
            'audio_clap_raw':    clp_raw.tolist(),
        })

    except Exception as e:
        errors.append({'song_id': row['song_id'], 'error': str(e)})
        continue

    # Checkpoint every N tracks
    if len(results) % CKPT_EVERY == 0:
        with open(CKPT_PATH, 'wb') as fh:
            pickle.dump(results, fh)

# Final checkpoint save
with open(CKPT_PATH, 'wb') as fh:
    pickle.dump(results, fh)

print(f'\nExtraction complete: {len(results)} succeeded, {len(errors)} failed')
if errors:
    print('Failed tracks:')
    for e in errors[:10]:
        print(f'  {e["song_id"]}: {e["error"]}')

## 5. Assemble and Save `audio_features.parquet`

In [ ]:
feat_df = pd.DataFrame(results)
print(f'Feature rows: {len(feat_df)}')
print(f'Columns: {feat_df.columns.tolist()}')

# Verify shapes before saving
hc_lens      = feat_df['audio_handcrafted'].apply(len)
clp_lens     = feat_df['audio_clap'].apply(len)
clp_raw_lens = feat_df['audio_clap_raw'].apply(len)
print(f'Handcrafted vector lengths — unique: {hc_lens.unique().tolist()}')
print(f'CLAP norm vector lengths   — unique: {clp_lens.unique().tolist()}')
print(f'CLAP raw vector lengths    — unique: {clp_raw_lens.unique().tolist()}')
assert hc_lens.eq(141).all(),  'Unexpected handcrafted vector length'
assert clp_lens.eq(512).all(), 'Unexpected CLAP normalized vector length'
assert clp_raw_lens.eq(512).all(), 'Unexpected CLAP raw vector length'

# Merge with quadrant / AV labels for convenience in later phases
meta_cols = df[['song_id', 'quadrant', 'arousal', 'valence']]
feat_df   = feat_df.merge(meta_cols, on='song_id', how='left')

feat_df.to_parquet(OUT_PATH, index=False, engine='pyarrow')
size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f'\nSaved: {OUT_PATH}  ({size_mb:.1f} MB)')

## 6. Validation

In [ ]:
# Reload and validate
feat = pd.read_parquet(OUT_PATH)
print(f'Loaded: {feat.shape}')
print()

# Check for NaNs (can arise from silent or corrupt audio)
hc_arr      = np.vstack(feat['audio_handcrafted'].values)
clp_arr     = np.vstack(feat['audio_clap'].values)
clp_raw_arr = np.vstack(feat['audio_clap_raw'].values)

hc_nan_rows      = np.isnan(hc_arr).any(axis=1).sum()
clp_nan_rows     = np.isnan(clp_arr).any(axis=1).sum()
clp_raw_nan_rows = np.isnan(clp_raw_arr).any(axis=1).sum()
print(f'Handcrafted NaN rows:  {hc_nan_rows} / {len(feat)}')
print(f'CLAP norm NaN rows:    {clp_nan_rows} / {len(feat)}')
print(f'CLAP raw NaN rows:     {clp_raw_nan_rows} / {len(feat)}')

# Value ranges and norm statistics
clp_norms     = np.linalg.norm(clp_arr, axis=1)
clp_raw_norms = np.linalg.norm(clp_raw_arr, axis=1)
print(f'\nHandcrafted range: [{hc_arr.min():.3f}, {hc_arr.max():.3f}]')
print(f'CLAP norm norms:   mean={clp_norms.mean():.6f}  std={clp_norms.std():.6f}  (should be 1.0)')
print(f'CLAP raw norms:    mean={clp_raw_norms.mean():.4f}  std={clp_raw_norms.std():.4f}  min={clp_raw_norms.min():.4f}  max={clp_raw_norms.max():.4f}')
print(f'  audio_clap is unit-normalized:     {np.allclose(clp_norms, 1.0, atol=1e-5)}')
print(f'  audio_clap_raw is NOT unit-normalized: {not np.allclose(clp_raw_norms, 1.0, atol=1e-2)}')

# Verify that normalizing clp_raw recovers clp
sample = clp_raw_arr[:5]
recovered = sample / np.linalg.norm(sample, axis=1, keepdims=True)
match = np.allclose(recovered, clp_arr[:5], atol=1e-5)
print(f'  normalize(clap_raw) == clap_norm (first 5 tracks): {match}')

# Coverage
print(f'\nCoverage by quadrant:')
print(feat.groupby('quadrant')['song_id'].count().sort_index().to_string())

## 7. Sanity Check — Do Features Separate Quadrants?

A quick PCA projection of the hand-crafted vectors, colored by emotion quadrant.  
If the features carry any signal, Q1–Q4 should show some separation even in 2D.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

QUADRANT_COLORS = {'Q1': '#f4a261', 'Q2': '#e63946', 'Q3': '#457b9d', 'Q4': '#2a9d8f'}
QUADRANT_LABELS = {'Q1': 'Happy', 'Q2': 'Angry', 'Q3': 'Sad', 'Q4': 'Calm'}

def pca_scatter(ax, X, labels, title):
    X_scaled = StandardScaler().fit_transform(X)
    pcs       = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
    for q, color in QUADRANT_COLORS.items():
        mask = labels == q
        ax.scatter(pcs[mask, 0], pcs[mask, 1], c=color, s=8, alpha=0.45,
                   label=f'{q} {QUADRANT_LABELS[q]}')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.legend(fontsize=8, markerscale=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('PCA of Audio Features — Quadrant Separation Sanity Check',
             fontsize=13, fontweight='bold')

pca_scatter(axes[0], hc_arr,  feat['quadrant'], 'Hand-crafted features (141-dim → PC1/PC2)')
pca_scatter(axes[1], clp_arr, feat['quadrant'], 'CLAP embeddings (512-dim → PC1/PC2)')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'audio_pca.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: audio_pca.png')

## 8. Feature Importance — Which Dimensions Drive Quadrant Separation?

Train a quick Random Forest classifier on hand-crafted features to rank feature group importance.  
This is interpretability analysis, not part of the recommendation pipeline.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

X = StandardScaler().fit_transform(hc_arr)
y = feat['quadrant'].values

clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f'Quadrant classification accuracy (5-fold CV):')
print(f'  Hand-crafted features: {scores.mean():.3f} ± {scores.std():.3f}')
print(f'  Chance level (4 classes): 0.250')

# Feature group importances
clf.fit(X, y)
imp = clf.feature_importances_

# Map dimension indices to feature group names
feature_groups = (
    [f'MFCC_mean_{i}' for i in range(40)] +
    [f'MFCC_std_{i}'  for i in range(40)] +
    [f'Chroma_mean_{i}' for i in range(12)] +
    [f'Chroma_std_{i}'  for i in range(12)] +
    [f'Tonnetz_mean_{i}' for i in range(6)] +
    [f'Tonnetz_std_{i}'  for i in range(6)] +
    ['Centroid_mean', 'Centroid_std',
     'BW_mean', 'BW_std',
     'Rolloff_mean', 'Rolloff_std'] +
    [f'Contrast_mean_{i}' for i in range(7)] +
    [f'Contrast_std_{i}'  for i in range(7)] +
    ['ZCR_mean', 'ZCR_std',
     'RMS_mean', 'RMS_std',
     'Tempo']
)

group_names = ['MFCC', 'Chroma', 'Tonnetz', 'Centroid', 'BW', 'Rolloff', 'Contrast', 'ZCR', 'RMS', 'Tempo']
group_bounds = [0, 80, 104, 116, 118, 120, 122, 136, 138, 140, 141]

group_imp = {}
for i, name in enumerate(group_names):
    group_imp[name] = imp[group_bounds[i]:group_bounds[i+1]].sum()

fig, ax = plt.subplots(figsize=(9, 4))
names  = list(group_imp.keys())
values = list(group_imp.values())
bars = ax.barh(names[::-1], values[::-1], color='#457b9d', alpha=0.85)
ax.set_xlabel('Total feature importance (Random Forest)')
ax.set_title('Which acoustic feature groups best separate emotion quadrants?',
             fontsize=11, fontweight='bold')
for bar, val in zip(bars, values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'audio_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: audio_feature_importance.png')

## Phase 2 Complete

**Output:** `outputs/audio_features.parquet`

Columns: `song_id`, `audio_handcrafted` (141-dim list), `audio_clap` (512-dim list, L2-normalized),
`audio_clap_raw` (512-dim list, un-normalized), `quadrant`, `arousal`, `valence`

| Column | Dims | Normalized | Use |
|---|---|---|---|
| `audio_handcrafted` | 141 | No (natural scale) | Interpretable ablation |
| `audio_clap` | 512 | Yes (norm=1.0) | Audio-only FAISS index; late fusion |
| `audio_clap_raw` | 512 | No (varying norm) | Early fusion concatenation with `lyrics_sbert_raw` |

**Why two CLAP columns?**  
`get_audio_features()` returns L2-normalized embeddings. If both CLAP and SBERT are unit-normalized before concatenation, `[c; s]` re-normalizes to `[c/√2; s/√2]`, making early fusion score = 0.5·cosine_audio + 0.5·cosine_lyrics = late fusion at α=0.5. Storing the raw projection before the `/norm` step lets Phase 4 build a non-trivial early fusion index.

**Loading in later phases:**
```python
feat = pd.read_parquet('outputs/audio_features.parquet')
hc_arr      = np.vstack(feat['audio_handcrafted'].values)  # (N, 141)
clp_arr     = np.vstack(feat['audio_clap'].values)         # (N, 512), unit-normalized
clp_raw_arr = np.vstack(feat['audio_clap_raw'].values)     # (N, 512), un-normalized
```

**Next:** Phase 3 — Lyrical Feature Extraction (`phase3_lyrics_features.ipynb`)